# Lazy Cell DIVE OME-TIFF review and tumor annotation

This notebook implements the local-review/server-import workflow for Cell DIVE-style single-channel OME-TIFFs:

1. Select one or more channel files.
2. Open each OME-TIFF lazily with `tiffslide -> zarr -> xarray`. Each selected channel becomes a separate SpatialData image element and therefore a separate napari layer.
3. Optionally add the pipeline's whole-cell and nuclear uint32 TIFF masks as lazy multiscale label elements.
4. Draw one polygon per tumor in napari-spatialdata.
5. Export the tumor polygons as GeoJSON in intrinsic full-resolution pixel coordinates.
6. Copy only the GeoJSON back to the server and import it into the canonical SpatialData store.

The source TIFF handles must remain open while napari is running. Keep the `open_slides` variable alive until the viewer is closed. The raw channel TIFFs and masks must share the same full-resolution canvas and origin; the validation cell fails loudly if their shapes differ.

## Environment

Run this in the modern SpatialData environment, not the InstanSeg/Nimbus environment. A typical local installation is:

```bash
python -m pip install "napari-spatialdata[all]" tiffslide xarray dask zarr geopandas shapely tifffile
```

If Qt event-loop integration is unreliable inside JupyterLab, launch the notebook from a terminal in the same environment. The viewer call blocks until napari closes; this is useful because the following export cell runs only after the in-memory annotation has returned to `sdata`.

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Mapping, Sequence

import dask.array as da
import geopandas as gpd
import numpy as np
import spatialdata
import tiffslide
import xarray as xr
from napari_spatialdata import Interactive
from spatialdata import SpatialData, read_zarr
from spatialdata.models import Image2DModel, Labels2DModel, ShapesModel
from spatialdata.transformations import Identity, Scale, set_transformation
from xarray import DataTree, Dataset

print({
    "spatialdata": spatialdata.__version__,
    "tiffslide": tiffslide.__version__,
    "xarray": xr.__version__,
})

## Local inputs

Use short, unique channel aliases. The dictionary keys become both SpatialData element names and napari layer names. Explicit paths are preferred over filename parsing because Cell DIVE filenames vary and marker aliases are part of the experimental metadata.

`CHANNELS_TO_LOAD = None` loads every entry; otherwise list just the channels needed for the review. Set either mask path to `None` to omit it. Pipeline defaults are `<slide>_whole_cell.tiff` and `<slide>_nuclear.tiff`.

In [ ]:
SLIDE_ID = "SLIDE-XXXX"

CHANNEL_FILES: dict[str, Path] = {
    "DAPI": Path(r"/path/to/SLIDE-XXXX_DAPI.ome.tif"),
    "PanCK": Path(r"/path/to/SLIDE-XXXX_PanCK.ome.tif"),
    # "CD3": Path(r"/path/to/SLIDE-XXXX_CD3.ome.tif"),
}
CHANNELS_TO_LOAD: list[str] | None = ["DAPI", "PanCK"]

MASK_FILES: dict[str, Path | None] = {
    "cell_labels": Path(r"/path/to/masks_whole_cell/SLIDE-XXXX_whole_cell.tiff"),
    "nuclear_labels": Path(r"/path/to/masks_whole_cell/SLIDE-XXXX_nuclear.tiff"),
}

# Image, label, and locally drawn tumor geometries stay in full-resolution pixels.
LOCAL_COORDINATE_SYSTEM = "slide_pixels"

# Used later when pixel-coordinate GeoJSON is attached to the server's micron-scaled store.
PIXEL_SIZE_UM = 0.325

# Extra display levels for the single-level pipeline masks. Four factors give 1x, 2x, 4x, 8x, 16x.
MASK_SCALE_FACTORS: list[int] | None = [2, 2, 2, 2]

GEOJSON_PATH = Path(f"{SLIDE_ID}_tumor_regions.geojson")

## Lazy TIFF and SpatialData helpers

The image loader preserves the pyramid already present in each source OME-TIFF. It does not stack files into one shared pyramid, so channels with different numbers of levels remain valid. The label loader opens the full-resolution mask lazily and asks `Labels2DModel` to construct display levels without eagerly reading the complete mask.

In [ ]:
def _safe_element_name(name: str) -> str:
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", name.strip()).strip("._")
    if not safe or safe in {".", ".."} or safe.startswith("__"):
        raise ValueError(f"Invalid SpatialData element name derived from {name!r}.")
    return safe


def _level_keys(zarr_img: xr.Dataset) -> list[str]:
    multiscales = (zarr_img.attrs or {}).get("multiscales")
    if multiscales:
        keys = [str(item["path"]) for item in multiscales[0].get("datasets", [])]
        if keys:
            return keys
    keys = [str(key) for key in zarr_img.keys()]
    return sorted(keys, key=lambda value: (0, int(value)) if value.isdigit() else (1, value))


def _as_single_channel_yx(arr: xr.DataArray, *, path: Path, level_key: str) -> xr.DataArray:
    arr = arr.squeeze(drop=True)
    rename: dict[str, str] = {}
    for dim in arr.dims:
        dim_text = str(dim)
        if dim_text.lower() == "y" or dim_text.upper().startswith("Y"):
            rename[dim] = "y"
        elif dim_text.lower() == "x" or dim_text.upper().startswith("X"):
            rename[dim] = "x"
    if rename:
        arr = arr.rename(rename)
    if arr.ndim != 2 or set(arr.dims) != {"y", "x"}:
        raise ValueError(
            f"Expected one 2D channel in {path} level {level_key}; got dims={arr.dims}, shape={arr.shape}. "
            "Use one single-channel Cell DIVE OME-TIFF per CHANNEL_FILES entry."
        )
    return arr.transpose("y", "x")


def _open_tiffslide_zarr(path: Path) -> tuple[Any, xr.Dataset, list[str]]:
    if not path.exists():
        raise FileNotFoundError(path)
    slide = tiffslide.open_slide(str(path))
    zarr_img = xr.open_zarr(
        slide.zarr_group.store,
        consolidated=False,
        mask_and_scale=False,
    )
    keys = _level_keys(zarr_img)
    if not keys:
        slide.close()
        raise ValueError(f"No image levels found in {path}.")
    return slide, zarr_img, keys


def load_channel_element(
    path: Path,
    *,
    channel_name: str,
    coordinate_system: str,
) -> tuple[DataTree, Any, dict[str, Any]]:
    slide, zarr_img, keys = _open_tiffslide_zarr(path)
    nodes: dict[str, Dataset] = {}
    level_shapes: list[tuple[int, int]] = []
    for level_index, key in enumerate(keys):
        level_yx = _as_single_channel_yx(zarr_img[key], path=path, level_key=key)
        level_cyx = level_yx.expand_dims(c=[channel_name]).transpose("c", "y", "x")
        nodes[f"scale{level_index}"] = Dataset({"image": level_cyx})
        level_shapes.append(tuple(int(v) for v in level_yx.shape))

    tree = DataTree.from_dict(nodes)
    set_transformation(tree, {coordinate_system: Identity()}, set_all=True)
    Image2DModel.validate(tree)
    metadata = {
        "path": str(path.resolve()),
        "level_keys": keys,
        "level_shapes": level_shapes,
        "level_downsamples": [float(v) for v in slide.level_downsamples],
    }
    return tree, slide, metadata


def load_label_element(
    path: Path,
    *,
    coordinate_system: str,
    scale_factors: Sequence[int] | None,
) -> tuple[Any, Any, dict[str, Any]]:
    slide, zarr_img, keys = _open_tiffslide_zarr(path)
    base = _as_single_channel_yx(zarr_img[keys[0]], path=path, level_key=keys[0])
    parsed = Labels2DModel.parse(
        base.data,
        dims=("y", "x"),
        scale_factors=list(scale_factors) if scale_factors else None,
        transformations={coordinate_system: Identity()},
    )
    metadata = {
        "path": str(path.resolve()),
        "base_shape": tuple(int(v) for v in base.shape),
        "dtype": str(base.dtype),
        "source_level_count": len(keys),
    }
    return parsed, slide, metadata


def build_review_spatialdata(
    channel_files: Mapping[str, Path],
    *,
    channels_to_load: Sequence[str] | None = None,
    mask_files: Mapping[str, Path | None] | None = None,
    coordinate_system: str = "slide_pixels",
    mask_scale_factors: Sequence[int] | None = (2, 2, 2, 2),
) -> tuple[SpatialData, list[Any], dict[str, Any]]:
    selected = list(channel_files) if channels_to_load is None else list(channels_to_load)
    missing_aliases = [name for name in selected if name not in channel_files]
    if missing_aliases:
        raise KeyError(f"CHANNELS_TO_LOAD contains aliases absent from CHANNEL_FILES: {missing_aliases}")
    if not selected:
        raise ValueError("Select at least one image channel.")

    images: dict[str, Any] = {}
    labels: dict[str, Any] = {}
    handles: list[Any] = []
    details: dict[str, Any] = {"images": {}, "labels": {}}
    expected_shape: tuple[int, int] | None = None

    try:
        for alias in selected:
            element_name = _safe_element_name(alias)
            if element_name in images:
                raise ValueError(f"Duplicate SpatialData element name after sanitizing {alias!r}.")
            element, handle, metadata = load_channel_element(
                Path(channel_files[alias]),
                channel_name=alias,
                coordinate_system=coordinate_system,
            )
            handles.append(handle)
            base_shape = metadata["level_shapes"][0]
            if expected_shape is None:
                expected_shape = base_shape
            elif base_shape != expected_shape:
                raise ValueError(f"Channel {alias!r} shape {base_shape} != expected {expected_shape}.")
            images[element_name] = element
            details["images"][element_name] = metadata

        for label_name, raw_path in (mask_files or {}).items():
            if raw_path is None:
                continue
            safe_name = _safe_element_name(label_name)
            element, handle, metadata = load_label_element(
                Path(raw_path),
                coordinate_system=coordinate_system,
                scale_factors=mask_scale_factors,
            )
            handles.append(handle)
            if metadata["base_shape"] != expected_shape:
                raise ValueError(
                    f"Mask {label_name!r} shape {metadata['base_shape']} != image shape {expected_shape}. "
                    "Do not annotate until canvas size and origin are reconciled."
                )
            labels[safe_name] = element
            details["labels"][safe_name] = metadata

        details["canvas_shape_yx"] = expected_shape
        details["coordinate_system"] = coordinate_system
        return SpatialData(images=images, labels=labels), handles, details
    except Exception:
        for handle in handles:
            handle.close()
        raise

In [ ]:
sdata, open_slides, load_details = build_review_spatialdata(
    CHANNEL_FILES,
    channels_to_load=CHANNELS_TO_LOAD,
    mask_files=MASK_FILES,
    coordinate_system=LOCAL_COORDINATE_SYSTEM,
    mask_scale_factors=MASK_SCALE_FACTORS,
)

print(sdata)
print(json.dumps(load_details, indent=2, default=str))

## Open in napari-spatialdata

Each channel is a separate `sdata.images` element, so the loop below adds it as a separate napari image layer. Each mask is a labels layer. Adjust contrast, colormap, blending, visibility, and label opacity in napari as needed.

To annotate tumors:

1. Create one new **Shapes** layer.
2. Rename it exactly `tumor_regions`.
3. Use the polygon tool to draw one polygon per tumor. Multiple tumors belong in the same layer.
4. Select only that Shapes layer and press **Shift+E**. napari-spatialdata attaches it to the in-memory `sdata`.
5. Close napari so the next notebook cell can export the shapes.

Use polygons, not ellipse/circle tools; napari-spatialdata currently supports saving rectangles and polygons as polygon shapes.

In [ ]:
interactive = Interactive(sdata)
interactive.switch_coordinate_system(LOCAL_COORDINATE_SYSTEM)

for image_name in sdata.images:
    interactive.add_element(image_name, LOCAL_COORDINATE_SYSTEM)

for label_name in sdata.labels:
    interactive.add_element(label_name, LOCAL_COORDINATE_SYSTEM)
    label_layer = interactive.get_layer(label_name)
    if label_layer is not None:
        label_layer.opacity = 0.35
        label_layer.visible = label_name == "cell_labels"

interactive.run()

## Export tumor polygons to GeoJSON

This export deliberately stores geometry in intrinsic `(x, y)` full-resolution pixels. GeoJSON normally describes map coordinates, so the file includes explicit foreign metadata and per-feature properties declaring that these are pixel coordinates. Do not assign a geographic CRS.

If `tumor_regions` is absent, reopen the viewer and ensure the new Shapes layer was selected when **Shift+E** was pressed.

In [ ]:
TUMOR_LAYER_NAME = "tumor_regions"

if TUMOR_LAYER_NAME not in sdata.shapes:
    raise KeyError(
        f"{TUMOR_LAYER_NAME!r} is not in sdata.shapes. "
        "In napari, select the tumor Shapes layer, press Shift+E, and close the viewer."
    )

tumors = sdata.shapes[TUMOR_LAYER_NAME].copy().reset_index(drop=True)
if tumors.empty:
    raise ValueError("The tumor Shapes layer is empty.")
if not tumors.geometry.is_valid.all():
    bad = tumors.index[~tumors.geometry.is_valid].tolist()
    raise ValueError(f"Invalid tumor polygons at rows {bad}; repair them before export.")

tumors["tumor_id"] = [f"tumor_{i:03d}" for i in range(1, len(tumors) + 1)]
tumors["slide_id"] = SLIDE_ID
tumors["coordinate_units"] = "intrinsic_full_resolution_pixels"
tumors["pixel_size_um"] = float(PIXEL_SIZE_UM)
tumors = gpd.GeoDataFrame(tumors, geometry="geometry", crs=None)

geojson = json.loads(tumors.to_json(drop_id=True))
geojson["mif_pipeline"] = {
    "slide_id": SLIDE_ID,
    "coordinate_units": "intrinsic_full_resolution_pixels",
    "axis_order": "x_y",
    "canvas_shape_yx": list(load_details["canvas_shape_yx"]),
    "pixel_size_um": float(PIXEL_SIZE_UM),
}
GEOJSON_PATH.write_text(json.dumps(geojson, indent=2))
print(f"Wrote {len(tumors)} tumor polygons to {GEOJSON_PATH.resolve()}")

## Local round-trip check

Re-read the exact GeoJSON that will be copied to the server and add it back to the local object. This catches malformed files and coordinate mistakes before transfer. `tumor_regions_roundtrip` should overlay the polygons that were drawn.

In [ ]:
def read_pixel_geojson(path: Path) -> tuple[gpd.GeoDataFrame, dict[str, Any]]:
    payload = json.loads(path.read_text())
    metadata = payload.get("mif_pipeline", {})
    if metadata.get("coordinate_units") != "intrinsic_full_resolution_pixels":
        raise ValueError(f"GeoJSON does not declare intrinsic pixel coordinates: {metadata}")
    frame = gpd.GeoDataFrame.from_features(payload["features"], crs=None)
    if frame.empty:
        raise ValueError(f"No features found in {path}.")
    return frame, metadata


roundtrip_tumors, roundtrip_metadata = read_pixel_geojson(GEOJSON_PATH)
expected_canvas = tuple(load_details["canvas_shape_yx"])
file_canvas = tuple(roundtrip_metadata.get("canvas_shape_yx", ()))
if file_canvas != expected_canvas:
    raise ValueError(f"GeoJSON canvas {file_canvas} != loaded image canvas {expected_canvas}.")

sdata.shapes["tumor_regions_roundtrip"] = ShapesModel.parse(
    roundtrip_tumors,
    transformations={LOCAL_COORDINATE_SYSTEM: Identity()},
)
print(sdata.shapes["tumor_regions_roundtrip"][["tumor_id", "slide_id", "geometry"]])

## Server: import GeoJSON into the canonical SpatialData store

Run the remaining cells on the server after copying the GeoJSON there. The pipeline's finalized store maps raster pixel coordinates into `global` microns with `pixel_size_um`, so the pixel-coordinate tumor polygons receive the same `Scale` transformation. The geometry itself is not multiplied or rewritten.

The import checks the GeoJSON slide ID, pixel size, and canvas extent against the server store before writing. Replacement is opt-in. Back up the store before replacing an existing tumor element.

In [ ]:
SERVER_SPATIALDATA_PATH = Path(r"/server/path/to/SLIDE-XXXX/spatialdata.zarr")
SERVER_GEOJSON_PATH = Path(r"/server/path/to/SLIDE-XXXX_tumor_regions.geojson")
SERVER_SHAPES_NAME = "tumor_regions"
SERVER_COORDINATE_SYSTEM = "global"
OVERWRITE_SERVER_SHAPES = False

In [ ]:
def _base_raster_shape(element: Any) -> tuple[int, int]:
    if isinstance(element, DataTree):
        node = element["scale0"]
        arrays = list(node.data_vars.values())
        if len(arrays) != 1:
            raise ValueError(f"Expected one raster array at scale0; found {len(arrays)}.")
        arr = arrays[0]
    else:
        arr = element
    return tuple(int(v) for v in arr.shape[-2:])


def import_tumor_geojson_into_backed_sdata(
    store_path: Path,
    geojson_path: Path,
    *,
    shapes_name: str,
    expected_slide_id: str,
    pixel_size_um: float,
    coordinate_system: str = "global",
    overwrite: bool = False,
) -> SpatialData:
    if not store_path.exists():
        raise FileNotFoundError(store_path)
    if not geojson_path.exists():
        raise FileNotFoundError(geojson_path)

    frame, metadata = read_pixel_geojson(geojson_path)
    if metadata.get("slide_id") != expected_slide_id:
        raise ValueError(
            f"GeoJSON slide_id={metadata.get('slide_id')!r} != expected {expected_slide_id!r}."
        )
    file_pixel_size = float(metadata.get("pixel_size_um", np.nan))
    if not np.isclose(file_pixel_size, pixel_size_um):
        raise ValueError(f"GeoJSON pixel size {file_pixel_size} != server value {pixel_size_um}.")

    server_sdata = read_zarr(store_path)
    if not server_sdata.images:
        raise ValueError(f"No image element found in {store_path}.")
    reference_image_name = "full_image" if "full_image" in server_sdata.images else next(iter(server_sdata.images))
    server_canvas = _base_raster_shape(server_sdata.images[reference_image_name])
    file_canvas = tuple(int(v) for v in metadata.get("canvas_shape_yx", ()))
    if file_canvas != server_canvas:
        raise ValueError(
            f"GeoJSON canvas {file_canvas} != server image canvas {server_canvas}; "
            "the local TIFF and server store may not share an origin or resolution."
        )

    existing_on_disk = shapes_name in server_sdata.shapes and bool(server_sdata.locate_element(server_sdata.shapes[shapes_name]))
    if existing_on_disk and not overwrite:
        raise FileExistsError(
            f"{shapes_name!r} already exists in {store_path}. Set overwrite=True only after backing up the store."
        )
    if existing_on_disk:
        server_sdata.delete_element_from_disk(shapes_name)

    transform = Scale([pixel_size_um, pixel_size_um], axes=("y", "x"))
    server_sdata.shapes[shapes_name] = ShapesModel.parse(
        frame,
        transformations={coordinate_system: transform},
    )
    server_sdata.write_element(shapes_name, overwrite=False)
    return server_sdata


server_sdata = import_tumor_geojson_into_backed_sdata(
    SERVER_SPATIALDATA_PATH,
    SERVER_GEOJSON_PATH,
    shapes_name=SERVER_SHAPES_NAME,
    expected_slide_id=SLIDE_ID,
    pixel_size_um=PIXEL_SIZE_UM,
    coordinate_system=SERVER_COORDINATE_SYSTEM,
    overwrite=OVERWRITE_SERVER_SHAPES,
)
print(server_sdata)

## Cleanup

Close the TIFFSlide handles after the local viewer and GeoJSON export are finished. Re-run the build cell to reopen them for another napari session.

In [ ]:
for slide in open_slides:
    slide.close()
open_slides = []
print("Closed local TIFFSlide handles.")